In [7]:
# Libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten,Dense, Dropout)
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
tf.random.set_seed(42)

In [2]:
# Step 2: Load Data 
faces = fetch_olivetti_faces(shuffle=False)
X = faces.images      
y = faces.target      
print("Images:", X.shape)
print("Labels:", y.shape)
print("Number of people:", len(np.unique(y)))

Images: (400, 64, 64)
Labels: (400,)
Number of people: 40


In [12]:
loss, accuracy = 0.0000, 0.0000     # Model 1 test loss and accuracy from Part 2
loss2, accuracy2 = 0.0000, 0.0000   # Model 2 test loss and accuracy from Part 3

In [ ]:
# Step 14: Model comparison

print("Feature map differences")
print("-" * 60)
print("Model 1 (3x3 window) looks at a very small patch of the image, so its "
      "feature maps are sharper and more detailed, picking up fine edges such "
      "as the eyes, nose, mouth and hairline.")
print("Model 2 (9x9 window) looks at a much larger patch, so its feature maps "
      "are smoother and blurrier and respond to broader structures such as the "
      "overall face shape and large light and dark regions.")
print("In both models the pooling layer halves the maps from 64x64 to 32x32 "
      "and keeps the strongest activations, so the pooled maps look like "
      "smaller, coarser versions of the convolutional maps.")

print("\nPerformance on the test set")
print("-" * 60)
print(f"Model 1 (3x3): loss = {loss:.4f}, accuracy = {accuracy:.4f}")
print(f"Model 2 (9x9): loss = {loss2:.4f}, accuracy = {accuracy2:.4f}")

if accuracy > accuracy2:
    print("\nModel 1 gave better predictions. The 3x3 window learns fine facial "
          "detail with far fewer parameters (160 vs 1,312 in the convolutional "
          "layer), which suits a small training set of 200 images. The 9x9 "
          "window blurs detail that helps tell similar faces apart and is more "
          "prone to overfitting.")
elif accuracy2 > accuracy:
    print("\nModel 2 gave better predictions. The 9x9 window sees more of the "
          "face at once, so each filter captures broader context such as face "
          "shape and the relationship between features. The trade-off is more "
          "parameters (1,312 vs 160 in the convolutional layer) and blurrier "
          "maps that lose fine detail.")
else:
    print("\nBoth models reached the same test accuracy. Model 1 is the simpler "
          "choice because it has far fewer parameters (160 vs 1,312 in the "
          "convolutional layer).")

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)

y_train_cat = to_categorical(y_train, 40)
y_test_cat = to_categorical(y_test, 40)

X_train = X_train.reshape(-1, 64, 64, 1)
X_test = X_test.reshape(-1, 64, 64, 1)
idx = 0

In [ ]:
# Step 15: Model 3 - Optimized neural network
# Convolution Neural Network Model
input_tensor3 = Input(shape=(64, 64, 1))

x = Conv2D(32, (3, 3), strides=1, padding="same", activation="relu")(input_tensor3)
x = MaxPooling2D(pool_size=(2, 2), strides=2)(x)

x = Conv2D(64, (3, 3), strides=1, padding="same", activation="relu")(x)
x = MaxPooling2D(pool_size=(2, 2), strides=2)(x)

x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu", kernel_regularizer=l2(0.001))(x)
x = Dropout(0.3)(x)
output3 = Dense(40, activation="softmax")(x)

model3 = Model(inputs=input_tensor3, outputs=output3)

# Compiling model3
model3.compile(optimizer="adam",
               loss="categorical_crossentropy",
               metrics=["accuracy"])
model3.summary()

# Early stopping uses a validation split from the TRAINING data,
# so the test set stays unseen until the final evaluation
early_stop = EarlyStopping(monitor="val_loss", patience=8,
                           restore_best_weights=True)

history3 = model3.fit(X_train, y_train_cat,
                      epochs=60, batch_size=16,
                      validation_split=0.2,
                      callbacks=[early_stop], verbose=1)

# Print the accuracy on the test set
loss3, accuracy3 = model3.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nModel 3 test loss: {loss3:.4f}")
print(f"Model 3 test accuracy: {accuracy3:.4f}")

# Compare all three models
print("\nModel comparison (test set)")
print(f"Model 1 (3x3, 1 conv layer): loss = {loss:.4f},  accuracy = {accuracy:.4f}")
print(f"Model 2 (9x9, 1 conv layer): loss = {loss2:.4f}, accuracy = {accuracy2:.4f}")
print(f"Model 3 (optimized):         loss = {loss3:.4f}, accuracy = {accuracy3:.4f}")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 64, 64, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 64, 64, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 32, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 32, 32, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 16, 16, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 16384)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 16384)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 128)                 │       2,097,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 40)                  │           5,160 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,121,256 (8.09 MB)

 Trainable params: 2,121,256 (8.09 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 185ms/step - accuracy: 0.0250 - loss: 3.9809 - val_accuracy: 0.0000e+00 - val_loss: 3.8722
Epoch 2/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - accuracy: 0.0625 - loss: 3.8382 - val_accuracy: 0.0000e+00 - val_loss: 3.8603
Epoch 3/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.0625 - loss: 3.8125 - val_accuracy: 0.0250 - val_loss: 3.8518
Epoch 4/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 153ms/step - accuracy: 0.0688 - loss: 3.7753 - val_accuracy: 0.0000e+00 - val_loss: 3.8385
Epoch 5/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - accuracy: 0.0750 - loss: 3.7255 - val_accuracy: 0.0250 - val_loss: 3.8444
Epoch 6/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.0812 - loss: 3.6487 - val_accuracy: 0.0250 - val_loss: 3.8355
Epoch 7/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.1375 - loss: 3.5774 - val_accuracy: 0.0250 - val_loss: 3.8012
Epoch 8/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.2000 - loss: 3.3898 - val_ac

In [11]:
# Step 16: Reflection on Model 3

print("Reflection on Model 3")
print("-" * 60)
print("To optimize the model, I started from Model 1 and added a second "
      "convolutional and pooling block, increasing the filters to 32 and 64 so "
      "the network could learn simple features first and combine them into more "
      "complex facial features. I kept the 3x3 window because it keeps fine "
      "facial detail with fewer parameters, and I added a 128-neuron fully "
      "connected layer to give the network more capacity to separate the 40 "
      "people.")
print("Because there are only 200 training images, overfitting was the main "
      "risk, so I added L2 regularization, two dropout layers (0.5 and 0.3), "
      "and early stopping on a validation split taken from the training data, "
      "which kept the test set unseen until the final evaluation.")

print(f"\nTest accuracy: Model 1 = {accuracy:.4f}, Model 2 = {accuracy2:.4f}, "
      f"Model 3 = {accuracy3:.4f}")

if accuracy3 > max(accuracy, accuracy2):
    print("Model 3 beat both earlier models. The second convolutional block "
          "let the network build more complex features, while dropout, L2 and "
          "early stopping helped it generalize instead of memorizing the "
          "training images. On a small dataset, regularization matters as much "
          "as model size.")
else:
    print("Model 3 did not beat both earlier models on this run. With only 200 "
          "training images and a 200-image test set, results vary from run to "
          "run, and the added capacity can overfit even with regularization. "
          "Next I would try fewer filters, a lower dropout rate, or a smaller "
          "batch size, changing one setting at a time.")

Reflection on Model 3
------------------------------------------------------------
To optimize the model, I started from Model 1 and added a second convolutional and pooling block, increasing the filters to 32 and 64 so the network could learn simple features first and combine them into more complex facial features. I kept the 3x3 window because it keeps fine facial detail with fewer parameters, and I added a 128-neuron fully connected layer to give the network more capacity to separate the 40 people.
Because there are only 200 training images, overfitting was the main risk, so I added L2 regularization, two dropout layers (0.5 and 0.3), and early stopping on a validation split taken from the training data, which kept the test set unseen until the final evaluation.


NameError: name 'accuracy' is not defined